In [81]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader


torch.manual_seed(42)
np.random.seed(42)

In [82]:
database = pd.read_csv('/home/jaume/ConquerX/Mis_apuntes/Pytorch/ejercicio_redes/Pima_Indians_Diabetes.csv')

numero_ceros = (database[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']] == 0).sum(axis=0)

database[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']] = database[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']].replace(0,np.nan)

columnas = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']

# for col in columnas:
#     database[col] = database[col].fillna(database[col].median())

database[columnas] = database[columnas].fillna(database[columnas].median())


database.isna().sum()


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [83]:
database

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31,0
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.0,180.0,32.9,0.171,63,0
764,2,122.0,70.0,27.0,125.0,36.8,0.340,27,0
765,5,121.0,72.0,23.0,112.0,26.2,0.245,30,0
766,1,126.0,60.0,29.0,125.0,30.1,0.349,47,1


In [84]:
database['Outcome'].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [85]:
x_datos = database.drop(columns='Outcome')
y_datos = database['Outcome']

print(type(x_datos))
print(type(y_datos))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>


In [86]:
x_train,x_test,y_train,y_test = train_test_split(
    x_datos,y_datos,train_size=0.8,test_size=0.2,random_state=1,stratify=y_datos)

print(type(x_train))
print(type(x_test))
print(type(y_train))
print(type(y_test))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>


In [87]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [88]:
print(type(x_train))
print(type(x_test))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [89]:
t_x_train = torch.from_numpy(x_train).float().to('cpu')
t_x_test = torch.from_numpy(x_test).float().to('cpu')

t_y_train = torch.from_numpy(y_train.values).float().unsqueeze(1).to('cpu')
t_y_test = torch.from_numpy(y_test.values).float().unsqueeze(1).to('cpu')

print(f"x train: {t_x_train.shape}, x test: {t_x_test.shape} \t y train: {t_y_train.shape}, y test: {t_y_test.shape}")

x train: torch.Size([614, 8]), x test: torch.Size([154, 8]) 	 y train: torch.Size([614, 1]), y test: torch.Size([154, 1])


In [90]:
train_dataset = TensorDataset(t_x_train,t_y_train)
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)

train_dataset[0]

(tensor([-0.2530, -0.0345, -1.7255, -0.0176, -0.1711,  0.5021, -1.0477, -0.7003]),
 tensor([1.]))

In [91]:
n_entradas = t_x_train.shape[1]
print(n_entradas)

8


In [92]:
len(train_loader)

20

In [93]:
class Red(nn.Module):
    def __init__(self,n_entradas):
        super().__init__()

        self.capa1 = nn.Linear(n_entradas,6)
        self.capa2 = nn.Linear(6,4)
        self.capa3 = nn.Linear(4,1)

    def forward(self, inputs):

        pred1 = torch.relu(self.capa1(inputs))
        pred2 = torch.relu(self.capa2(pred1))
        pred_final = self.capa3(pred2)

        return pred_final

In [94]:
epochs = 100
lr = 0.0005
estatus_print = 10

In [95]:
model = Red(n_entradas=n_entradas)
criterion = nn.BCEWithLogitsLoss() # Contiene sigmoid
optimizer = optim.Adam(params=model.parameters(), lr = lr)


In [96]:
historico = []

for epoch in range(1,epochs +1):

    model.train()
    loss_acumulator = 0

    for x_batch, y_batch in train_loader:

        optimizer.zero_grad()

        y_pred = model(x_batch)
        loss = criterion(y_pred,y_batch)
        loss.backward()
        optimizer.step()
        loss_acumulator += loss.item()
    
    loss_acumulator = loss_acumulator / len(train_loader)

    model.eval()
    with torch.no_grad():
        # TEST
        y_pred_test = model(t_x_test)
        y_class_test = (y_pred_test >= 0).float()
        accuracy_test = ((y_class_test == t_y_test).float().mean()*100).round()

        # TRAIN
        y_pred_train = model(t_x_train)
        y_class_train = (y_pred_train >= 0).float()
        accuracy_train = ((y_class_train == t_y_train).float().mean()*100).round()

    if epoch % estatus_print == 0:
        print(f"Epoch: {epoch} \t Loss: {loss_acumulator:.6f}")
        print(f"Accuracy TEST: {accuracy_test} \t Accuracy TRAIN: {accuracy_train}")

    historico.append({
        'Epoch': epoch,
        'Loss' : loss_acumulator,
        'Acc.Test': accuracy_test,
        'Acc.Train': accuracy_train,
    })

historico = pd.DataFrame()

Epoch: 10 	 Loss: 0.695880
Accuracy TEST: 35.0 	 Accuracy TRAIN: 35.0
Epoch: 20 	 Loss: 0.639743
Accuracy TEST: 77.0 	 Accuracy TRAIN: 78.0
Epoch: 30 	 Loss: 0.552855
Accuracy TEST: 77.0 	 Accuracy TRAIN: 78.0
Epoch: 40 	 Loss: 0.486100
Accuracy TEST: 76.0 	 Accuracy TRAIN: 78.0
Epoch: 50 	 Loss: 0.452442
Accuracy TEST: 75.0 	 Accuracy TRAIN: 78.0
Epoch: 60 	 Loss: 0.455902
Accuracy TEST: 75.0 	 Accuracy TRAIN: 78.0
Epoch: 70 	 Loss: 0.439823
Accuracy TEST: 75.0 	 Accuracy TRAIN: 79.0
Epoch: 80 	 Loss: 0.441723
Accuracy TEST: 76.0 	 Accuracy TRAIN: 79.0
Epoch: 90 	 Loss: 0.433103
Accuracy TEST: 77.0 	 Accuracy TRAIN: 79.0
Epoch: 100 	 Loss: 0.429514
Accuracy TEST: 78.0 	 Accuracy TRAIN: 79.0
